# `pipeline_hopfield_v2` — Pipeline Completo com Classes `src/`

Reimplementação do fluxo de `script01_analises_preliminares.ipynb` usando as
classes disponíveis em `src/` em vez de funções inline.

**Contexto biológico.** O experimento parte de matrizes binárias de expressão
gênica (~40 000 células Fujita × ~45 000 células Mathys × N genes). São
selecionados os **5 000 genes mais frequentes** do Fujita e cada célula é
representada por um vetor SWeeP de **600 dimensões** via projeção `W0 · R5k`
(rSWeeP, AIBIALab). A memória associativa armazena perfis **binários** por
tipo celular; o espaço SWeeP é usado para clusterizar e escolher protótipos.

**Diferenças em relação ao `script01`:**
- Usa as classes de `src/` para binarização, alinhamento, seleção de genes,
  projeção SWeeP, extração de padrões e avaliação.
- Cobre o pipeline completo (binarização → avaliação cross-dataset).
- Remapeamento de classes seguindo o padrão do script01: classes não presentes
  em `[1, 3, 4, 5, 6, 7]` são remapeadas para a classe `2`.
- Configuração: `nc=10` clusters por classe, `k=1` representante por centroide
  → **70 padrões** (7 classes × 10 subclusters).

**Rede utilizada:** Modern Hopfield Network (Ramsauer et al., 2020) com
capacidade de armazenamento exponencial e recuperação equivalente a um passo
de *attention* (`softmax(β · Ξ · ξ) · Ξᵀ`).

## 1. Imports e configuração

In [1]:
import sys, os
import importlib
import numpy as np
import pandas as pd
import torch
import anndata as ad
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, f1_score, classification_report

SRC_DIR = os.path.join(os.path.dirname(os.path.abspath('__file__')), 'src')
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

import config
importlib.reload(config)
from config import (
    PATH_M, PATH_F, PATH_FEATURES_F, PATH_FEATURES_M,
    PATH_SWEEP_F, PATH_LABELS_F, PATH_LABELS_M,
    OUT_BINARIZACAO, OUT_ALINHAMENTO, OUT_TOP_GENES,
    OUT_TREINAMENTO, OUT_HOPFIELD,
)

from preprocessing import Binarizador
from alinhamento import (
    LeitorFeatures, AnalisadorSobreposicao, Alinhador,
    ValidadorAlinhamento, SelecionadorGenesFrequentes, AnalisadorCobertura,
)
from treinamento import (
    GeradorConjuntoTreinamento, CarregadorDadosFujita,
    ProjetorSWeP, ProjetorSWeePR,
    ExtratorPadroesSubcluster, ModernHopfieldNetwork, AvaliadorHopfield,
)
from treinamento.hopfield_utils import wsort, closervects

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    torch.cuda.manual_seed_all(SEED)

Dispositivo: cpu


## 2. Binarização

Converte as matrizes de expressão `.h5ad` para formato binário (valores > 0 → 1,
zeros → 0). O `Binarizador` detecta automaticamente se o arquivo já existe e
pula o processamento nesse caso.

In [2]:
binarizador_f = Binarizador(path_h5ad=PATH_F, out_dir=OUT_BINARIZACAO)
binarizador_m = Binarizador(path_h5ad=PATH_M, out_dir=OUT_BINARIZACAO)

binarizador_f.binarizar()
binarizador_m.binarizar()

print('Fujita binarizado em:', binarizador_f.path_binarizada)
print('Mathys binarizado em:', binarizador_m.path_binarizada)

[Binarizador] Arquivo já existe, pulando: c:\Users\Leticia\Documents\Letworkspace\pipiline_hopifield\outputs\binarizacao\matrizFiltradaeNormalizadaF\matrizBinarizadaM.h5ad
[Binarizador] Arquivo já existe, pulando: c:\Users\Leticia\Documents\Letworkspace\pipiline_hopifield\outputs\binarizacao\matrizFiltradaeNormalizadaMParcial\matrizBinarizadaM.h5ad
Fujita binarizado em: c:\Users\Leticia\Documents\Letworkspace\pipiline_hopifield\outputs\binarizacao\matrizFiltradaeNormalizadaF\matrizBinarizadaM.h5ad
Mathys binarizado em: c:\Users\Leticia\Documents\Letworkspace\pipiline_hopifield\outputs\binarizacao\matrizFiltradaeNormalizadaMParcial\matrizBinarizadaM.h5ad


## 3. Alinhamento de espaços gênicos

Os dois datasets têm espaços gênicos distintos (36 591 genes no Fujita,
32 643 no Mathys, ~30 312 em comum). O alinhamento:

1. Lê os mapeamentos `gene_name → Ensembl ID` de cada dataset.
2. Define a ordem canônica dos genes baseada no Fujita (referência).
3. Realinha ambas as matrizes para esse espaço canônico.
   - Genes ausentes no **Mathys** são preenchidos com `0.5` como sentinela.
4. Valida que as duas matrizes resultantes têm genes na mesma ordem.

In [3]:
# Passo 1 — Leitura dos arquivos de features
leitor = LeitorFeatures(PATH_FEATURES_F, PATH_FEATURES_M)
leitor.ler()
print(leitor)

[LeitorFeatures] Fujita : 36591 genes mapeados
[LeitorFeatures] Mathys : 32643 genes mapeados
LeitorFeatures(
  path_features_f = C:\Users\Leticia\Documents\Letworkspace\Sweep-Harmonization\Meus_testes\Controle_qualidade\dataF\dados_combinados\features.tsv\features.tsv
  path_features_m = C:\Users\Leticia\Documents\Letworkspace\Sweep-Harmonization\Meus_testes\Controle_qualidade\dataM\featuresM.tsv.gz
  map_f           = 36591 genes
  map_m           = 32643 genes
)


In [4]:
# Passo 2 — Análise de sobreposição dos espaços gênicos
# var_names idênticos no original e no binarizado — lemos direto do original
_f = ad.read_h5ad(PATH_F, backed='r')
var_names_f_original = _f.var_names.tolist()
_f.file.close()
del _f

analisador = AnalisadorSobreposicao(leitor.map_f, leitor.map_m, var_names_f_original)
analisador.analisar()
print(analisador)

[AnalisadorSobreposicao] Em comum  : 30312
[AnalisadorSobreposicao] Só Fujita : 6279
[AnalisadorSobreposicao] Só Mathys : 2331  ← serão excluídos
[AnalisadorSobreposicao] Espaço gênico final: 36601 genes
AnalisadorSobreposicao(
  ids_comuns      = 30312
  ids_so_f        = 6279
  ids_so_m        = 2331
  genes_ordenados = 36601
)


In [5]:
# Passo 3 — Alinhamento dos dois h5ad binarizados
alinhador = Alinhador(
    path_binarizada_m = binarizador_m.path_binarizada,
    path_binarizada_f = binarizador_f.path_binarizada,
    out_dir           = OUT_ALINHAMENTO,
    map_f             = leitor.map_f,
    map_m             = leitor.map_m,
    gene_alvo_idx     = analisador.gene_alvo_idx,
    genes_ordenados   = analisador.genes_ordenados,
)
alinhador.alinhar()
alinhador.salvar_como_txt()
alinhador.gerar_tracking(analisador.ids_so_f, leitor.map_f)
print(alinhador)

[Alinhador] Fujita já alinhado, pulando: c:\Users\Leticia\Documents\Letworkspace\pipiline_hopifield\outputs\alinhamento\adataF_binarizado_alinhado\adataF_binarizado_alinhado.h5ad
[Alinhador] Mathys já alinhado, pulando: c:\Users\Leticia\Documents\Letworkspace\pipiline_hopifield\outputs\alinhamento\adataM_binarizado_alinhado\adataM_binarizado_alinhado.h5ad

[Alinhador] Concluído.
[Alinhador] TXT já existe, pulando: c:\Users\Leticia\Documents\Letworkspace\pipiline_hopifield\outputs\alinhamento\adataF_binarizado_alinhado\adataF_binarizado_alinhado.txt
[Alinhador] TXT já existe, pulando: c:\Users\Leticia\Documents\Letworkspace\pipiline_hopifield\outputs\alinhamento\adataM_binarizado_alinhado\adataM_binarizado_alinhado.txt
[Alinhador] Tracking já existe, pulando: c:\Users\Leticia\Documents\Letworkspace\pipiline_hopifield\outputs\alinhamento\tracking_genes_adicionados_mathys.csv
Alinhador(
  path_binarizada_m = c:\Users\Leticia\Documents\Letworkspace\pipiline_hopifield\outputs\binarizacao\ma

In [6]:
# Passo 4 — Validação da ordem de genes
validador = ValidadorAlinhamento(
    path_f_alinhado = alinhador.path_f_alinhado,
    path_m_alinhado = alinhador.path_m_alinhado,
    genes_ordenados = analisador.genes_ordenados,
)
validador.validar()

[ValidadorAlinhamento] Carregando metadados...
✓ Número de genes idêntico: 36601
✓ Fujita alinhado == ordem de referência
✓ Mathys alinhado == ordem de referência
✓ Fujita alinhado == Mathys alinhado
[ValidadorAlinhamento] Validação concluída com sucesso.


ValidadorAlinhamento(
  path_f_alinhado  = c:\Users\Leticia\Documents\Letworkspace\pipiline_hopifield\outputs\alinhamento\adataF_binarizado_alinhado\adataF_binarizado_alinhado.h5ad
  path_m_alinhado  = c:\Users\Leticia\Documents\Letworkspace\pipiline_hopifield\outputs\alinhamento\adataM_binarizado_alinhado\adataM_binarizado_alinhado.h5ad
  genes_ordenados  = 36601 genes
)

## 4. Seleção dos top-5000 genes frequentes

Seleciona os 5 000 genes com maior frequência (soma de coluna) no Fujita.
Em seguida verifica quantos desses genes estão presentes no Mathys e gera
os conjuntos filtrados para treinamento.

In [7]:
path_top5k   = os.path.join(OUT_TOP_GENES,   'top5000_frequentes.csv')
path_f_top5k = os.path.join(OUT_TREINAMENTO, 'adataF_binarizado_alinhado_top5000.txt')
path_m_top5k = os.path.join(OUT_TREINAMENTO, 'adataM_binarizado_alinhado_top5000.txt')

path_f_txt = alinhador.path_f_alinhado.replace('.h5ad', '.txt')
path_m_txt = alinhador.path_m_alinhado.replace('.h5ad', '.txt')

# Top 5000 genes mais frequentes do Fujita
selecionador = SelecionadorGenesFrequentes(path_txt=path_f_txt, n=5000)
selecionador.calcular(out_csv=path_top5k).salvar(path_top5k)
print(selecionador)

[SelecionadorGenesFrequentes] Lendo: c:\Users\Leticia\Documents\Letworkspace\pipiline_hopifield\outputs\alinhamento\adataF_binarizado_alinhado\adataF_binarizado_alinhado.txt
  Calculando frequências para 36600 genes (streaming)...


: 

In [ ]:
# Cobertura dos top-5000 genes do Fujita no Mathys
cobertura = AnalisadorCobertura(path_top5k, leitor.map_f, leitor.map_m)
cobertura.analisar(out_csv=os.path.join(OUT_ALINHAMENTO, 'top5000_cobertura_mathys.csv'))

In [ ]:
# Conjuntos de treinamento filtrados (Fujita + Mathys)
gerador = GeradorConjuntoTreinamento(
    path_top_genes_csv = path_top5k,
    out_dir            = OUT_TREINAMENTO,
)
gerador.gerar(path_f_txt)
gerador.gerar(path_m_txt)
print(gerador)

## 5. Projeção SWeeP (rSWeeP via R / fallback Python)

Projeta a matriz binarizada do Fujita (células × 5 000 genes) no espaço
SWeeP de 600 dimensões usando a base ortonormal rSWeeP.
Se o R não estiver disponível, `ProjetorSWeePR` usa o fallback Python (QR sintético).

```
Wswp = W0 @ R5k        (células × 600)
```

In [ ]:
projetor_r = ProjetorSWeePR(
    path_matriz   = path_f_top5k,
    path_saida    = PATH_SWEEP_F,
    n_componentes = 600,
    seed          = SEED,
)
projetor_r.projetar()

## 6. Carregamento dos dados

Carrega:
- `W0`: matriz binária Fujita (células × 5 000 genes) — usada como padrões da rede.
- `labels`: rótulos inteiros de tipo celular por célula.
- `Wswp`: projeções SWeeP pré-computadas (células × 600) — usadas para K-means.

In [ ]:
# Fujita — padrões de treinamento
carregador = CarregadorDadosFujita(
    path_matriz = path_f_top5k,
    path_genes  = path_top5k,
    path_labels = PATH_LABELS_F,
    path_sweep  = PATH_SWEEP_F,
    n_genes     = 5000,
)
carregador.carregar()
print(carregador)

In [ ]:
# Mathys — dados para imputação cross-dataset
# Genes ausentes no Mathys foram preenchidos com 0.5 (sentinela) pelo Alinhador.
print('[Mathys] Carregando matriz top5000...')
W_mathys = pd.read_csv(path_m_top5k).to_numpy(dtype=np.float32)
print(f'[Mathys] W_mathys shape: {W_mathys.shape}')

print('[Mathys] Carregando rótulos...')
labels_mathys = np.loadtxt(PATH_LABELS_M, dtype=int)
print(f'[Mathys] labels shape: {labels_mathys.shape}, tipos: {np.unique(labels_mathys)}')

## 7. Remapeamento de classes (clo)

Seguindo o padrão do `script01_analises_preliminares.m` original:
classes não presentes em `[1, 3, 4, 5, 6, 7]` são remapeadas para `2`,
resultando em 7 classes: Excitatory (1), Endothelial/remapeadas (2),
Inhibitory (3), Astrocytes (4), Microglia (5), Oligodendrocytes (6),
OPCs (7).

```matlab
clo = cl;
clo(~ismember(clo,[1 3 4 5 6 7 0])) = 2;
```

In [ ]:
clo = carregador.labels.copy()
clo[~np.isin(clo, [1, 3, 4, 5, 6, 7, 0])] = 2

clo_m = labels_mathys.copy()
clo_m[~np.isin(clo_m, [1, 3, 4, 5, 6, 7, 0])] = 2

print('Distribuição Fujita (clo):')
vals, counts = np.unique(clo, return_counts=True)
for v, c in zip(vals, counts):
    print(f'  classe {v}: {c:>6d} células')

print('\nDistribuição Mathys (clo_m):')
vals_m, counts_m = np.unique(clo_m, return_counts=True)
for v, c in zip(vals_m, counts_m):
    print(f'  classe {v}: {c:>6d} células')

## 8. PCA no espaço SWeeP

Aplica PCA **sem centralização** sobre as projeções SWeeP — equivalente ao
`pca(W, 'Centered', false)` do MATLAB. Os scores resultantes `Wpc`
são usados como espaço auxiliar para visualizações e análises.

In [ ]:
projetor = ProjetorSWeP(n_features=5000, n_componentes=600, seed=SEED)
projetor.usar_sweep_precomputado(carregador.Wswp).aplicar_pca()
print(projetor)

## 9. Extração de padrões por subcluster (perf35)

Para cada uma das 7 classes executa KMeans com `nc=10` clusters no espaço
SWeeP e seleciona o vetor binário mais próximo de cada centroide como
representante. Resulta em `7 × 10 = 70 padrões`.

```matlab
for ii in classes:
    km = kmeans(Wswp[clo==ii], nc)
    for centroide in km.centroids:
        idx = closervects(Wswp[clo==ii], centroide, k=1)
        perf35.append(W0[clo==ii][idx])
```

In [ ]:
extrator = ExtratorPadroesSubcluster(
    W0      = carregador.W0,
    labels  = clo,
    classes = [1, 2, 3, 4, 5, 6, 7],
    nc      = 10,
    seed    = SEED,
    k       = 1,
)
extrator.extrair(projetor.Wswp)
perf35 = extrator.padroes
print(extrator)
print(f'perf35 shape: {perf35.shape}  (esperado: (70, 5000))')

## 10. Treinamento da rede (rede35)

Armazena os 70 padrões na Modern Hopfield Network.

**Regra de armazenamento:** simplesmente guardar os padrões — não há
treinamento iterativo.

**Parâmetros:**
- `beta=8.0`: temperatura inversa do softmax (maior → mais winner-takes-all)
- `n_iters=1`: uma iteração de atualização já é suficiente
- `threshold=0.5`: genes preenchidos com 0.5 (sentinela Mathys) são tratados
  como ausentes (< 0.5 → 0) na binarização da query

In [ ]:
rede35 = ModernHopfieldNetwork(beta=8.0, n_iters=1, binary=True, threshold=0.5)
rede35.store(perf35)
meta_eval = extrator.meta  # mapeamento padrao -> classe
print(rede35)

## 10b. Alternativa: carregar rede pre-treinada

**Use esta celula em vez das secoes 9 e 10** quando o treinamento foi feito
noutra maquina. Copie os arquivos rede35_v2.pt e rede35_v2_metadata.json para
a pasta outputs/hopfield/ desta maquina e execute apenas esta celula.

Se treinou aqui (secoes 9 e 10), **pule esta celula**.

In [ ]:
import json as _json

PATH_PT   = os.path.join(OUT_HOPFIELD, 'rede35_v2.pt')
PATH_META = os.path.join(OUT_HOPFIELD, 'rede35_v2_metadata.json')

# Carrega rede e metadados
rede35 = ModernHopfieldNetwork.carregar(PATH_PT)

with open(PATH_META) as _f:
    _meta_json = _json.load(_f)

# Reconstrói perf35 em {0,1} a partir dos padrões salvos em {-1,+1}
perf35 = ((rede35.patterns.cpu().numpy() + 1.0) / 2.0).clip(0.0, 1.0).astype('float32')

# Variáveis de avaliação (substituem extrator.meta quando rede vem de fora)
meta_eval = [tuple(x) for x in _meta_json['meta']]

print(rede35)
print(f'perf35 shape: {perf35.shape}')
print(f'Classes: {_meta_json["classes"]}  nc={_meta_json["nc"]}  padroes={_meta_json["n_patterns"]}')

## 11. Teste numa subclasse (clo == 3)

Seguindo o padrão da seção 13 do `script01`: embaralha aleatoriamente as
células da classe 3 e testa as primeiras 1000 (amostra representativa).

```matlab
Wk4  = wsort(W0(clo==3, :));          % embaralhamento aleatório
Wtes = hopf_ts(Wk4(1:1000,:), rede35);
```

In [ ]:
NC   = 10
CLASSES_ARR = np.array([1, 2, 3, 4, 5, 6, 7])

Wk4    = wsort(carregador.W0[clo == 3])
n_test = min(1000, Wk4.shape[0])
Wtes   = rede35.retrieve(Wk4[:n_test], batch_size=4096)
print(f'hopf_ts(Wk4[:{n_test}], rede35): shape {Wtes.shape}')

perf35_f = perf35.astype(np.float64)
Wtes_f   = Wtes.astype(np.float64)
a2 = (Wtes_f ** 2).sum(axis=1, keepdims=True)
b2 = (perf35_f ** 2).sum(axis=1, keepdims=True).T
idx_proto = (a2 + b2 - 2 * (Wtes_f @ perf35_f.T)).argmin(axis=1)
pred_sub  = CLASSES_ARR[idx_proto // NC]

acc_sub = (pred_sub == 3).mean()
print(f'\nAcurácia subclasse clo==3: {acc_sub * 100:.2f}%')

In [ ]:
y_true_sub = np.full(n_test, 3)
labels_sub = sorted(set(y_true_sub) | set(pred_sub))
print(classification_report(y_true_sub, pred_sub, labels=labels_sub, zero_division=0))

cm_sub = confusion_matrix(y_true_sub, pred_sub, labels=labels_sub)
fig, ax = plt.subplots(figsize=(max(6, len(labels_sub)), max(5, len(labels_sub))))
sns.heatmap(cm_sub, annot=True, fmt='d', cmap='Oranges',
            xticklabels=labels_sub, yticklabels=labels_sub, ax=ax)
ax.set_xlabel('Predito'); ax.set_ylabel('Real')
ax.set_title('Matriz de Confusão — rede35 (subconjunto clo==3)')
plt.tight_layout(); plt.show()

## 12. Auto-imputação — Fujita → Fujita

Baseline interno: a rede treinada em Fujita recebe as próprias células Fujita.
Esperamos alta taxa de reconstrução e classificação.

In [ ]:
print('=== Auto-imputação: Fujita → Fujita ===')
Wrecuperado_f = rede35.retrieve(carregador.W0, batch_size=4096)

avaliador_f = AvaliadorHopfield(
    padroes = perf35,
    classes = [1, 2, 3, 4, 5, 6, 7],
    nc      = 10,
    meta    = meta_eval,
)
avaliador_f.avaliar(Wrecuperado_f, clo).plotar(titulo='Confusão — rede35 (Fujita → Fujita)')
print(avaliador_f)

## 13. Imputação cross-dataset — Mathys com sentinela 0.5

A rede treinada em Fujita recebe células do Mathys. Os 6 289 genes ausentes
no Mathys foram preenchidos com `0.5` pelo `Alinhador` — o limiar `threshold=0.5`
da rede os trata como ausentes (0) na binarização da query antes da recuperação.

In [ ]:
print('=== Imputação cross-dataset: Mathys (com sentinela 0.5) ===')
Wrecuperado_m = rede35.retrieve(W_mathys, batch_size=4096)

avaliador_m = AvaliadorHopfield(
    padroes = perf35,
    classes = [1, 2, 3, 4, 5, 6, 7],
    nc      = 10,
    meta    = meta_eval,
)
avaliador_m.avaliar(Wrecuperado_m, clo_m).plotar(titulo='Confusão — rede35 (Mathys → Fujita, 0.5)')
print(avaliador_m)

## 14. Imputação cross-dataset — Mathys binário puro (0.5 → 0)

Comparação: converte os valores sentinela `0.5 → 0` antes da recuperação,
equivalente a tratar todos os genes ausentes como definitivamente inativos.
Permite comparar o impacto do sentinela `0.5` na qualidade da recuperação.

In [ ]:
W_mathys_bin = W_mathys.copy()
n_meio = int((W_mathys_bin == 0.5).sum())
W_mathys_bin[W_mathys_bin == 0.5] = 0.0
print(f'Valores convertidos de 0.5 → 0: {n_meio}')
print(f'Valores únicos após conversão: {np.unique(W_mathys_bin)}')

print('\n=== Imputação cross-dataset: Mathys (binário puro, sem 0.5) ===')
Wrecuperado_m_bin = rede35.retrieve(W_mathys_bin, batch_size=4096)

avaliador_m_bin = AvaliadorHopfield(
    padroes = perf35,
    classes = [1, 2, 3, 4, 5, 6, 7],
    nc      = 10,
    meta    = meta_eval,
)
avaliador_m_bin.avaliar(Wrecuperado_m_bin, clo_m).plotar(titulo='Confusão — rede35 (Mathys binário puro)')
print(avaliador_m_bin)

## 15. Persistência da rede

In [ ]:
import json as _json

os.makedirs(OUT_HOPFIELD, exist_ok=True)
path_rede_v2 = os.path.join(OUT_HOPFIELD, 'rede35_v2.pt')
path_meta_v2 = os.path.join(OUT_HOPFIELD, 'rede35_v2_metadata.json')

# Salva rede
rede35.salvar(path_rede_v2)

# Salva metadados (necessários para AvaliadorHopfield na máquina de aplicação)
_metadata = {
    'classes'   : [1, 2, 3, 4, 5, 6, 7],
    'nc'        : 10,
    'n_patterns': int(perf35.shape[0]),
    'n_genes'   : int(perf35.shape[1]),
    'meta'      : [[int(c_), int(idx)] for c_, idx in extrator.meta],
}
with open(path_meta_v2, 'w') as _f:
    _json.dump(_metadata, _f, indent=2)

print(f'Rede salva em    : {path_rede_v2}')
print(f'Metadados salvos : {path_meta_v2}')
print('Para carregar: rede = ModernHopfieldNetwork.carregar(path)')

## Notas finais

**Diferenças em relação ao `pipilinePrincipal.ipynb`:**
- Inclui remapeamento de classes `clo` (padrão do script01): classes raras → 2.
- Inclui teste em subclasse (seção 11) e auto-imputação Fujita→Fujita (seção 12).
- Avalia tanto Mathys com sentinela `0.5` quanto Mathys binário puro.

**Hiperparâmetros:**
- `beta`: controla a nitidez da recuperação. Para padrões esparsos de alta
  dimensão, valores entre 4 e 16 costumam funcionar bem.
- `nc`: número de subclusters por classe. Aumentar aumenta a capacidade de
  representar variabilidade intraclasse.
- `k`: número de representantes por centroide. `k=1` usa o indivíduo mais
  próximo; `k>1` pode ser usado para padrões de consenso.

**Classes utilizadas (src/):**
```
preprocessing/  → Binarizador
alinhamento/    → LeitorFeatures, AnalisadorSobreposicao, Alinhador,
                   ValidadorAlinhamento, SelecionadorGenesFrequentes, AnalisadorCobertura
treinamento/    → GeradorConjuntoTreinamento, CarregadorDadosFujita,
                   ProjetorSWeP, ProjetorSWeePR, ExtratorPadroesSubcluster,
                   ModernHopfieldNetwork, AvaliadorHopfield
```

## 16. Análise Comparativa — Fujita vs Mathys

Consolida as matrizes de confusão e as métricas de reconstrução dos três cenários
avaliados nas seções 12, 13 e 14 para facilitar a comparação direta.

In [ ]:
# Matrizes de confusão — contagens e normalizadas (Fujita + Mathys lado a lado)
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

avaliador_f.plotar(titulo='Fujita → Fujita (contagens)',    ax=axes[0, 0])
avaliador_f.plotar(titulo='Fujita → Fujita (normalizada)',  ax=axes[0, 1], normalizado=True)
avaliador_m.plotar(titulo='Mathys → Fujita (contagens)',    ax=axes[1, 0])
avaliador_m.plotar(titulo='Mathys → Fujita (normalizada)',  ax=axes[1, 1], normalizado=True)

plt.suptitle('Matrizes de Confusão — rede35', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Tabela de métricas globais — comparação entre os três cenários
df_resumo = pd.DataFrame([
    avaliador_f.metricas_resumo('Fujita → Fujita'),
    avaliador_m.metricas_resumo('Mathys → Fujita (0.5)'),
    avaliador_m_bin.metricas_resumo('Mathys → Fujita (bin)'),
]).set_index('dataset')

display(df_resumo.style
    .format({
        'n_celulas':         '{:,}',
        'acuracia':          '{:.2%}',
        'f1_macro':          '{:.4f}',
        'f1_weighted':       '{:.4f}',
        'taxa_reconstrucao': '{:.2%}',
        'semelhanca_media':  '{:.4f}',
    })
    .set_caption('Métricas de Reconstrução — rede35')
    .highlight_max(color='lightgreen', axis=0)
)

In [ ]:
# Métricas por classe — precision, recall e F1 para cada dataset
print('=== Fujita → Fujita ===')
display(avaliador_f.metricas_por_classe())

print('=== Mathys → Fujita (0.5) ===')
display(avaliador_m.metricas_por_classe())

print('=== Mathys → Fujita (binário puro) ===')
display(avaliador_m_bin.metricas_por_classe())